# APDTM vs CASH: Fair Leave-One-Log-Out Comparison

This notebook contains the comparison between APDTM and our CASH recommender. The goal is to separate several effects that are otherwise mixed together: the learning model, the feature set, the training data, the set of algorithms that may be recommended, and whether hyperparameters are considered.

The evaluation protocol is leave-one-real-log-out. In each fold, one real event log is held out for testing. Training data from the same log family is removed, including augmented logs derived from the held-out real log.


## What This Notebook Does

The notebook reads the prepared APDTM and CASH result tables, checks whether all 11 comparison setups have already been evaluated, and then displays the relevant summary tables for reporting.

If the expected outputs are already present, the notebook does not retrain the models. If outputs are missing, it calls `apdtm_comparison/fair_lolo_ablation.py` to recreate only the missing setups.

The main result for the paper is the table **Mean Accuracy Over All Weight Scenarios**. It reports the same kind of min-max recommendation accuracy that is used in the ProReco-vs-CASH evaluation.


## Method

APDTM and CASH solve related but different recommendation tasks. APDTM is an algorithm selector: it predicts one of five discovery algorithms (`AM`, `HM`, `IM`, `IMf`, `IMd`) using APDTM log meta-features. CASH is a configuration selector: it predicts the expected quality of algorithm and hyperparameter candidates and then recommends the candidate with the best predicted composite score.

To make the comparison interpretable, the setups below vary one dimension at a time:

- **model:** APDTM-style `RandomForestClassifier` versus CASH-style `RandomForestRegressor`
- **features:** APDTM meta-features versus CASH log features
- **data:** original APDTM data, real CASH logs, synthetic logs, and leakage-clean full CASH data
- **action space:** APDTM-compatible five algorithms versus all CASH algorithms
- **hyperparameters:** default representatives versus observed tuned configurations

For default-only candidates, measured `v6_baseline_*` rows from `dataset_v8.csv` are preferred. If no measured baseline row exists for a log and algorithm, the script falls back to parameterless Alpha rows or to the nearest documented/project default.


## Data Used

The comparison uses these inputs:

- `apdtm_comparison/data/dataset_v8.csv`: observed CASH results for log, algorithm, hyperparameters, and quality metrics
- `apdtm_comparison/vendor/process_discovery_meta_learning/log_meta_features.csv`: original APDTM meta-feature table
- `apdtm_comparison/vendor/process_discovery_meta_learning/discovery_metrics.csv`: original APDTM discovery metrics
- `apdtm_comparison/outputs/apdtm_cash_real_log_meta_features.csv`: APDTM features extracted for our real CASH logs
- `apdtm_comparison/outputs/apdtm_cash_real_discovery_metrics.csv`: APDTM default discovery metrics on our real CASH logs

The output tables are stored in `apdtm_comparison/outputs/fair_lolo_ablation/`.


In [1]:
from pathlib import Path
import pandas as pd


def find_project_root(start: Path) -> Path:
    """Find the repository root without hard-coding a local absolute path."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "apdtm_comparison" / "data" / "dataset_v8.csv").exists() and (candidate / "apdtm_comparison" / "fair_lolo_ablation.py").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing apdtm_comparison/data/dataset_v8.csv and apdtm_comparison/fair_lolo_ablation.py")


PROJECT_ROOT = find_project_root(Path.cwd())
APDTM_DIR = PROJECT_ROOT / "apdtm_comparison"
OUTPUT_DIR = APDTM_DIR / "outputs" / "fair_lolo_ablation"
SCRIPT = APDTM_DIR / "fair_lolo_ablation.py"


def rel(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT))


print("PROJECT_ROOT = .")
print("SCRIPT =", rel(SCRIPT))
print("OUTPUT_DIR =", rel(OUTPUT_DIR))


PROJECT_ROOT = .
SCRIPT = apdtm_comparison/fair_lolo_ablation.py
OUTPUT_DIR = apdtm_comparison/outputs/fair_lolo_ablation


## Run Or Load Evaluation

This cell checks whether all comparison setups are already available in `lolo_summary.csv`. When the outputs are complete, it only loads them. When some setups are missing, it reruns the missing setups with `fair_lolo_ablation.py`.

This makes the notebook fast to open for reporting, while still keeping the evaluation reproducible.


In [2]:
import subprocess, sys

expected_setups = {
    "B0_apdtm_cls_apdtm_features_original",
    "B1_apdtm_cls_apdtm_features_original_plus_cash_real",
    "B2_apdtm_cls_cash_features_cash_real_default",
    "B3_apdtm_cls_cash_features_cash_real_synthetic_default",
    "B4_apdtm_cls_cash_features_cash_full_clean_default",
    "C0_cash_reg_apdtm_features_cash_real_default",
    "C1_cash_reg_cash_features_cash_real_default",
    "C2_cash_reg_cash_features_cash_full_clean_apdtm5_default",
    "C3_cash_reg_cash_features_cash_full_clean_apdtm5_hparams",
    "C4_cash_reg_cash_features_cash_full_clean_all_algos_default",
    "C5_cash_reg_cash_features_cash_full_clean_all_algos_hparams",
}
summary_path = OUTPUT_DIR / "lolo_summary.csv"
if summary_path.exists():
    existing_summary = pd.read_csv(summary_path)
    existing_setups = set(existing_summary["setup_id"].dropna().unique())
else:
    existing_setups = set()

if expected_setups.issubset(existing_setups):
    print(f"Existing complete outputs found in {rel(OUTPUT_DIR)}; skipping retraining.")
else:
    missing = sorted(expected_setups - existing_setups)
    print("Missing setups, running training:", missing)
    cmd = [sys.executable, str(SCRIPT), "--output-dir", str(OUTPUT_DIR), "--only", *missing]
    print("Running:", " ".join([Path(cmd[0]).name, rel(SCRIPT), "--output-dir", rel(OUTPUT_DIR), "--only", *missing]))
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr)
    completed.check_returncode()


Existing complete outputs found in apdtm_comparison/outputs/fair_lolo_ablation; skipping retraining.


## Load Result Tables

The result CSVs contain one row per setup, held-out log, and metric-weight scenario. The summary table aggregates these rows into mean min-max accuracies, regrets, and ranks.


In [3]:
summary = pd.read_csv(OUTPUT_DIR / "lolo_summary.csv")
results = pd.read_csv(OUTPUT_DIR / "lolo_results.csv")
models = pd.read_csv(OUTPUT_DIR / "model_manifest.csv")
setups = pd.read_csv(OUTPUT_DIR / "setup_manifest.csv")

print("rows", len(results), "setups", results["setup_id"].nunique(), "logs", results["cash_log_id"].nunique())

summary_equal = summary[summary["weights"].eq("equal")].copy()
summary_mean = summary[summary["weights"].eq("MEAN")].copy()

for frame in (summary_equal, summary_mean):
    frame["model"] = frame["setup_id"].str.extract(r"^([BC]\d)")
    frame["candidate_space"] = frame["action_space"].map({
        "apdtm5_default": "APDTM-5 defaults",
        "apdtm5_hparams": "CASH hparams on APDTM-compatible algorithms",
        "cash_all_default": "all CASH defaults",
        "cash_all_hparams": "all CASH algorithms + hparams",
    })
    frame["training_data"] = frame["data_regime"].map({
        "apdtm_original": "APDTM original",
        "apdtm_original_plus_cash_real": "APDTM original + real CASH logs",
        "cash_real": "real CASH logs",
        "cash_real_synthetic": "real + synthetic CASH logs",
        "cash_full": "Full CASH dataset",
    })
    frame["features"] = frame["feature_set"].map({"apdtm": "APDTM", "cash": "CASH"})
    frame["recommender"] = frame["model_kind"].map({"classifier": "RF classifier", "regressor": "RF regressor"})

display_cols = [
    "model",
    "recommender",
    "features",
    "training_data",
    "candidate_space",
    "n_logs",
    "mean_accuracy_full",
    "mean_regret_full",
    "mean_train_instances",
]

main_lolo_accuracy = summary_equal[display_cols].sort_values("model").reset_index(drop=True)
main_lolo_accuracy_rounded = main_lolo_accuracy.copy()
for col in ["mean_accuracy_full", "mean_regret_full", "mean_train_instances"]:
    main_lolo_accuracy_rounded[col] = main_lolo_accuracy_rounded[col].round(3)

main_lolo_accuracy_rounded


rows 2880 setups 11 logs 18


,model,recommender,features,training_data,candidate_space,n_logs,mean_accuracy_full,mean_regret_full,mean_train_instances
0,B0,RF classifier,APDTM,APDTM original,APDTM-5 defaults,16,0.543,0.361,800.812
1,B1,RF classifier,APDTM,APDTM original + real CASH logs,APDTM-5 defaults,16,0.767,0.191,815.812
2,B2,RF classifier,CASH,real CASH logs,APDTM-5 defaults,18,0.821,0.145,16.056
3,B3,RF classifier,CASH,real + synthetic CASH logs,APDTM-5 defaults,18,0.827,0.140,78.056
4,B4,RF classifier,CASH,Full CASH dataset,APDTM-5 defaults,18,0.829,0.138,131.222
5,C0,RF regressor,APDTM,real CASH logs,APDTM-5 defaults,16,0.610,0.304,60.938
6,C1,RF regressor,CASH,real CASH logs,APDTM-5 defaults,18,0.829,0.138,85.000
7,C2,RF regressor,CASH,Full CASH dataset,APDTM-5 defaults,18,0.817,0.149,990.500
8,C3,RF regressor,CASH,Full CASH dataset,CASH hparams on APDTM-compatible algorithms,18,0.945,0.045,17188.500
9,C4,RF regressor,CASH,Full CASH dataset,all CASH defaults,18,0.817,0.149,1982.000


## Equal-Weight Accuracy

This table uses only the equal composite weighting: fitness, precision, generalization, and simplicity each receive weight `0.25`.

`mean_accuracy_full` is the mean min-max recommendation accuracy over the full candidate space of each held-out log:

`(chosen_score - worst_score) / (best_score - worst_score)`

This value is useful for intuition, but it is not the main paper number because the ProReco-vs-CASH evaluation averages over multiple metric-weight scenarios.


In [4]:
main_lolo_accuracy_rounded


,model,recommender,features,training_data,candidate_space,n_logs,mean_accuracy_full,mean_regret_full,mean_train_instances
0,B0,RF classifier,APDTM,APDTM original,APDTM-5 defaults,16,0.543,0.361,800.812
1,B1,RF classifier,APDTM,APDTM original + real CASH logs,APDTM-5 defaults,16,0.767,0.191,815.812
2,B2,RF classifier,CASH,real CASH logs,APDTM-5 defaults,18,0.821,0.145,16.056
3,B3,RF classifier,CASH,real + synthetic CASH logs,APDTM-5 defaults,18,0.827,0.140,78.056
4,B4,RF classifier,CASH,Full CASH dataset,APDTM-5 defaults,18,0.829,0.138,131.222
5,C0,RF regressor,APDTM,real CASH logs,APDTM-5 defaults,16,0.610,0.304,60.938
6,C1,RF regressor,CASH,real CASH logs,APDTM-5 defaults,18,0.829,0.138,85.000
7,C2,RF regressor,CASH,Full CASH dataset,APDTM-5 defaults,18,0.817,0.149,990.500
8,C3,RF regressor,CASH,Full CASH dataset,CASH hparams on APDTM-compatible algorithms,18,0.945,0.045,17188.500
9,C4,RF regressor,CASH,Full CASH dataset,all CASH defaults,18,0.817,0.149,1982.000


## Mean Accuracy Over All Weight Scenarios

This is the main table for the paper. It averages each setup over all 15 metric-weight scenarios used in the evaluation grid. This makes it directly comparable to the ProReco-vs-CASH evaluation, which also reports mean min-max accuracy over logs and weightings.

The most important column is `mean_accuracy_full`. It evaluates every recommendation on the full observed candidate space of the held-out log. The `candidate_space` column shows what each setup was allowed to choose from.


In [5]:
mean_lolo_accuracy = (
    summary_mean[display_cols]
    .sort_values("mean_accuracy_full", ascending=False)
    .reset_index(drop=True)
)

mean_lolo_accuracy_rounded = mean_lolo_accuracy.copy()
for col in ["mean_accuracy_full", "mean_regret_full", "mean_train_instances"]:
    mean_lolo_accuracy_rounded[col] = mean_lolo_accuracy_rounded[col].round(3)

mean_lolo_accuracy_rounded


,model,recommender,features,training_data,candidate_space,n_logs,mean_accuracy_full,mean_regret_full,mean_train_instances
0,C5,RF regressor,CASH,Full CASH dataset,all CASH algorithms + hparams,18,0.941,0.053,38818.000
1,C3,RF regressor,CASH,Full CASH dataset,CASH hparams on APDTM-compatible algorithms,18,0.926,0.066,17188.500
2,C4,RF regressor,CASH,Full CASH dataset,all CASH defaults,18,0.819,0.159,1982.000
3,C2,RF regressor,CASH,Full CASH dataset,APDTM-5 defaults,18,0.810,0.166,990.500
4,C1,RF regressor,CASH,real CASH logs,APDTM-5 defaults,18,0.808,0.167,85.000
5,B4,RF classifier,CASH,Full CASH dataset,APDTM-5 defaults,18,0.782,0.191,131.222
6,B3,RF classifier,CASH,real + synthetic CASH logs,APDTM-5 defaults,18,0.780,0.194,78.056
7,B2,RF classifier,CASH,real CASH logs,APDTM-5 defaults,18,0.775,0.199,16.056
8,B1,RF classifier,APDTM,APDTM original + real CASH logs,APDTM-5 defaults,16,0.724,0.246,815.812
9,C0,RF regressor,APDTM,real CASH logs,APDTM-5 defaults,16,0.632,0.313,60.938


## Saved Models

The script stores one final model artifact per setup in `outputs/fair_lolo_ablation/models/`. These artifacts are trained on the available data for the setup after the LOLO evaluation has been completed. They are mainly kept for reproducibility and inspection; the reported results come from the LOLO folds.


In [6]:
models_display = models.copy()
models_display["model_path"] = models_display["model_path"].map(lambda x: rel(PROJECT_ROOT / x) if not Path(str(x)).is_absolute() else rel(Path(x)))
models_display


,setup_id,model_path,n_final_train_instances,mean_train_instances
0,B0_apdtm_cls_apdtm_features_original,apdtm_comparison 5/outputs/fair_lolo_ablation/...,801,800.812500
1,B1_apdtm_cls_apdtm_features_original_plus_cash...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,817,815.812500
2,B2_apdtm_cls_cash_features_cash_real_default,apdtm_comparison 5/outputs/fair_lolo_ablation/...,17,16.055556
3,B3_apdtm_cls_cash_features_cash_real_synthetic...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,79,78.055556
4,B4_apdtm_cls_cash_features_cash_full_clean_def...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,135,131.222222
5,C0_cash_reg_apdtm_features_cash_real_default,apdtm_comparison 5/outputs/fair_lolo_ablation/...,65,60.937500
6,C1_cash_reg_cash_features_cash_real_default,apdtm_comparison 5/outputs/fair_lolo_ablation/...,90,85.000000
7,C2_cash_reg_cash_features_cash_full_clean_apdt...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,1013,990.500000
8,C3_cash_reg_cash_features_cash_full_clean_apdt...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,17580,17188.500000
9,C4_cash_reg_cash_features_cash_full_clean_all_...,apdtm_comparison 5/outputs/fair_lolo_ablation/...,2027,1982.000000


## Interpretation For The Paper

The APDTM comparison should be read in two steps.

First, the ablation rows show which design choices matter. `B0` and `B1` are APDTM-style classifier baselines. `B2` to `B4` show APDTM-style classification with CASH features and data. `C0` to `C2` show the CASH regressor under restricted/default conditions. These rows isolate the effects of features, data, and model class.

Second, the CASH-delta rows show the additional value of the larger CASH task. `C2` keeps the APDTM-compatible algorithms at default settings, `C3` uses CASH to choose among observed hyperparameter configurations for those APDTM-compatible algorithm families, `C4` allows all CASH algorithms at default settings, and `C5` allows all observed CASH algorithms and hyperparameters. This mirrors the structure of the ProReco-vs-CASH comparison: first compare under a restricted menu, then quantify the gain from the full CASH search space.
